In [1]:
from google.colab import drive
# 运行后会弹窗要求你授权登录谷歌账号
drive.mount('/content/drive')
print("云盘挂载成功！你的文件将保存在 /content/drive/MyDrive/ 下")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
云盘挂载成功！你的文件将保存在 /content/drive/MyDrive/ 下


In [2]:
!ls /content/drive/MyDrive/

'Colab Notebooks'   hf_cache   LaTeX_OCR_train	 mydoc	 zoom.txt


In [4]:
# %%bash
# # 卸载可能冲突的旧包
# pip uninstall -y unsloth transformers trl peft accelerate

In [5]:
%%bash
# 安装 Unsloth 官方推荐的 Colab 依赖
pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-o23e9z9w/unsloth_3326f05145934ef7a87c29d051e95850
  Resolved https://github.com/unslothai/unsloth.git to commit 4c72e09480d5f0c21d6739c62b23e7d501464ffc
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-o23e9z9w/unsloth_3326f05145934ef7a87c29d051e95850


In [6]:
%%bash
# 2. 强制升级 trl 到 0.9.0 以上，并同步升级相关核心库
pip install "trl==0.24.0" peft accelerate bitsandbytes "datasets==4.3.0"
pip install datasets pillow pillow-heif

In [7]:
import torch
from unsloth import FastVisionModel
from datasets import load_dataset
from transformers import TextStreamer
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
# ==========================================
# 1. 初始化模型与 Tokenizer
# ==========================================
# Colab 在海外，直接使用 Unsloth 官方优化的 4bit 模型，免去手动下载和全精度转换的麻烦
model_name = "unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name,
    load_in_4bit=True,                         # 在 Colab T4 GPU 上必须开启 4bit 以防止显存溢出 (OOM)
    use_gradient_checkpointing="unsloth",
)


==((====))==  Unsloth 2026.6.9: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.90G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

In [9]:
# ==========================================
# 2. 设置 LoRA / Peft 配置
# ==========================================
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


In [10]:
# ==========================================
# 3. 数据集载入与格式转换
# ==========================================

# 替换为你实际的数据集路径
# 这里以 Hugging Face 上的开源数据集为例
dataset_path = "linxy/LaTeX_OCR"
dataset = load_dataset(dataset_path, split="train")

def convert_to_conversation(sample):
    instruction = "Write the LaTeX representation for this image."
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": sample["image"]}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": sample["text"]}
            ]
        }
    ]
    return {"messages": conversation}

# 转换整个数据集的格式以符合 SFTTrainer 的要求
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

print(f"数据集载入成功！共包含 {len(converted_dataset)} 条训练数据。")
print("第一条数据样例预览：")
print(converted_dataset[0])

README.md:   0%|          | 0.00/5.73k [00:00<?, ?B/s]

full/train-00000-of-00001.parquet:   0%|          | 0.00/383M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/76318 [00:00<?, ? examples/s]

数据集载入成功！共包含 76318 条训练数据。
第一条数据样例预览：
{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'Write the LaTeX representation for this image.'}, {'type': 'image', 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=500x100 at 0x7903B53B87A0>}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': 'd s ^ { 2 } = ( 1 - { \\frac { q c o s \\theta } { r } } ) ^ { \\frac { 2 } { 1 + \\alpha ^ { 2 } } } \\lbrace d r ^ { 2 } + r ^ { 2 } d \\theta ^ { 2 } + r ^ { 2 } s i n ^ { 2 } \\theta d \\varphi ^ { 2 } \\rbrace - { \\frac { d t ^ { 2 } } { ( 1 - { \\frac { q c o s \\theta } { r } } ) ^ { \\frac { 2 } { 1 + \\alpha ^ { 2 } } } } } .'}]}]}


In [11]:
# ==========================================
# 4. 配置训练器并开始微调 (SFT)
# ==========================================
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=converted_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=30,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",               # 8bit 优化器，Colab 必备
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="/content/outputs",    # 训练中间结果存放在临时盘即可
        report_to="none",

        # 关键修复：显式禁用 bf16 并确保 fp16 开启
        fp16=True,
        bf16=False,

        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)

print("\n--- 开始微调训练 ---")
trainer_stats = trainer.train()

Unsloth: Model does not have a default image size - using 512

--- 开始微调训练 ---


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 76,318 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 51,521,536 of 8,343,688,192 (0.62% trained)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a futur

Step,Training Loss
1,3.526841
2,3.485474
3,3.144254
4,3.106345
5,1.738658
6,1.624517
7,1.112588
8,1.187450
9,1.142271
10,0.962414


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in /content/outputs/checkpoint-30/tokenizer_config.json.


In [12]:

# ==========================================
# 5. 模型保存 (直接保存到你的 Google Drive，防止丢失)
# ==========================================
# 将路径设置为挂载好的谷歌云盘中
save_path = "/content/drive/MyDrive/Qwen2.5-VL-LaTeX-Finetuned"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"\n模型已安全保存至你的谷歌云盘：{save_path}")


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Qwen2.5-VL-LaTeX-Finetuned/tokenizer_config.json.



模型已安全保存至你的谷歌云盘：/content/drive/MyDrive/Qwen2.5-VL-LaTeX-Finetuned


In [13]:
# ==========================================
# 6. 推理测试
# ==========================================
print("\n--- 开始使用微调后的模型进行测试推理 ---")

FastVisionModel.for_inference(model)

# 抽取第3张图测试
test_image = dataset[2]["image"]
instruction = "Write the LaTeX representation for this image."



--- 开始使用微调后的模型进行测试推理 ---


In [14]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": instruction}
        ]
    }
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

inputs = tokenizer(
    test_image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=1.5,
    min_p=0.1
)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-

\mu _ { L } ( q ) = \sum _ { m = 1 } ^ { L } P _ { L } ( m ) ~ \frac { 1 } { q ^ { m - 1 } } .<|im_end|>
